# BDC Epistolary Graph Construction

Builds the unweighted undirected epistolary graph G = (L, E) over all BDC letter nodes.

**Edge criterion (conjunctive):**  
An edge (li, lj) exists iff:
1. li and lj share at least one non-Bullinger person ref (sender or recipient), **AND**
2. |date_i − date_j| ≤ 30 days 


In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import time
from pathlib import Path
from collections import defaultdict

print('pandas :', pd.__version__)
print('networkx:', nx.__version__)

META_CSV     = Path('bdc_metadata.csv')
OUT_GRAPHML  = Path('bdc_graph.graphml')
OUT_EDGELIST = Path('bdc_graph_edges.csv')

BULLINGER   = 'p495'
TEMP_WINDOW = 30

pandas : 3.0.1
networkx: 3.6.1


## 1. Load metadata

In [2]:
def normalise_date_str(s):
    if not s or isinstance(s, float): return None
    s = str(s).strip()
    if '/' in s: s = s.split('/')[0].strip()
    if not s: return None
    if len(s) == 4 and s.isdigit(): return f'{s}-01-01'
    if len(s) == 7: return f'{s}-01'
    return s

df = pd.read_csv(META_CSV)
df['date_parsed'] = pd.to_datetime(
    df['date_when'].apply(normalise_date_str),
    format='mixed', dayfirst=False, errors='coerce'
).astype('datetime64[us]')

print(f'Loaded        : {len(df)} letters')
print(f'Dates resolved: {df["date_parsed"].notna().sum()}')
df[['letter_id','sender_ref','recipient_ref','date_when','date_parsed']].head(4)

Loaded        : 13114 letters
Dates resolved: 13114


,letter_id,sender_ref,recipient_ref,date_when,date_parsed
0,file1,p1283,p495,1547,1547-01-01
1,file10,p8049,p495,1548-01-12,1548-01-12
2,file100,p396,p495,1548-04-23,1548-04-23
3,file1000,p20172,p495,1550-08-22,1550-08-22


## 2. Build node set

In [3]:
node_attrs = {}
for _, row in df.iterrows():
    lid = str(row['letter_id'])
    node_attrs[lid] = {
        'sender_ref'     : str(row['sender_ref'])     if pd.notna(row['sender_ref'])     else '',
        'sender_cert'    : str(row['sender_cert'])    if pd.notna(row['sender_cert'])    else '',
        'recipient_ref'  : str(row['recipient_ref'])  if pd.notna(row['recipient_ref'])  else '',
        'recipient_cert' : str(row['recipient_cert']) if pd.notna(row['recipient_cert']) else '',
        'date_str'       : str(row['date_when'])      if pd.notna(row['date_when'])      else '',
        'date_precision' : str(row['date_precision']) if pd.notna(row['date_precision']) else '',
        'date_cert'      : str(row['date_cert'])      if pd.notna(row['date_cert'])      else '',
        'place_ref'      : str(row['place_ref'])      if pd.notna(row['place_ref'])      else '',
        'language'       : str(row['language'])       if pd.notna(row['language'])       else '',
        # enrichment placeholders
        'dominant_topic' : '',
        'topic_dist'     : '',
        'patristic_hits' : '',
    }
print(f'Nodes: {len(node_attrs)}')

Nodes: 13114


## 3. Build inverted correspondent index

Maps each non-Bullinger person ref to a date-sorted list of (date, letter_id) tuples.
Only letters with full or range date precision are eligible — year-only and year-month
dates cannot meaningfully participate in a 30-day window.

In [4]:
eligible = set(
    df.loc[
        df['date_parsed'].notna() &
        df['date_precision'].isin(['full', 'range']),
        'letter_id'
    ].astype(str)
)
print(f'Letters eligible for edges : {len(eligible)} / {len(df)}')

date_lookup = dict(zip(df['letter_id'].astype(str), df['date_parsed']))

person_index = defaultdict(list)
for _, row in df.iterrows():
    lid = str(row['letter_id'])
    if lid not in eligible:
        continue
    for role in ['sender_ref', 'recipient_ref']:
        ref = row[role]
        if pd.notna(ref) and ref != BULLINGER:
            person_index[ref].append((date_lookup[lid], lid))

for ref in person_index:
    person_index[ref].sort(key=lambda x: x[0])

bucket_sizes = pd.Series([len(v) for v in person_index.values()])
print(f'Non-Bullinger refs indexed : {len(person_index)}')
print(f'Bucket size — median: {bucket_sizes.median():.0f}, '
      f'p90: {bucket_sizes.quantile(0.9):.0f}, '
      f'max: {bucket_sizes.max()}')
print(f'Single-letter persons (no edges): {(bucket_sizes == 1).sum()}')

Letters eligible for edges : 12858 / 13114
Non-Bullinger refs indexed : 956
Bucket size — median: 2, p90: 22, max: 758
Single-letter persons (no edges): 414


## 4. Generate edges

Sliding window per person bucket: for each letter, connect all letters
by the same correspondent within the 30-day window

In [5]:
edge_set  = set()
window_td = np.timedelta64(TEMP_WINDOW, 'D')

t0 = time.time()
for ref, entries in person_index.items():
    dates = [e[0] for e in entries]
    lids  = [e[1] for e in entries]
    n     = len(entries)
    left  = 0
    for right in range(1, n):
        while (dates[right] - dates[left]) > window_td:
            left += 1
        for i in range(left, right):
            li, lj = (lids[i], lids[right]) if lids[i] < lids[right] \
                     else (lids[right], lids[i])
            edge_set.add((li, lj))

print(f'Edges generated: {len(edge_set):,}  ({time.time()-t0:.1f}s)')

Edges generated: 21,610  (0.1s)


## 5. Assemble NetworkX graph

In [6]:
G = nx.Graph()
for lid, attrs in node_attrs.items():
    G.add_node(lid, **attrs)

for (li, lj) in edge_set:
    di, dj = date_lookup.get(li), date_lookup.get(lj)
    if pd.notna(di) and pd.notna(dj):
        date_min = str(min(di, dj))[:10]
    else:
        date_min = str(di)[:10] if pd.notna(di) else str(dj)[:10]
    G.add_edge(li, lj, date_min=date_min)

print(f'Nodes    : {G.number_of_nodes():,}')
print(f'Edges    : {G.number_of_edges():,}')
print(f'Isolated : {sum(1 for n in G.nodes if G.degree(n) == 0):,}')

Nodes    : 13,114
Edges    : 21,610
Isolated : 4,518


## 8. Write files

In [9]:
nx.write_graphml(G, OUT_GRAPHML)
print(f'GraphML : {OUT_GRAPHML}  ({OUT_GRAPHML.stat().st_size/1e6:.1f} MB)')

edges_df = pd.DataFrame(
    [{'source': u, 'target': v, 'date_min': d['date_min']}
     for u, v, d in G.edges(data=True)]
)
edges_df.to_csv(OUT_EDGELIST, index=False)
print(f'Edge CSV: {OUT_EDGELIST}  ({len(edges_df):,} edges)')

GraphML : bdc_graph.graphml  (6.8 MB)
Edge CSV: bdc_graph_edges.csv  (21,610 edges)
